<a href="https://colab.research.google.com/github/jlurena/csci635_final_project/blob/initial/wip.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import glob
import numpy as np
import pandas as pd

In [ ]:
#dataset- https://physionet.org/content/challenge-2012/get-zip/1.0.0/

In [ ]:
DATA_DIR = "data/physionet2012"
SETS = ["set-a", "set-b", "set-c"]
OUTCOMES = {
    "set-a": os.path.join(DATA_DIR, "Outcomes-a.txt"),
    "set-b": os.path.join(DATA_DIR, "Outcomes-b.txt"),
    "set-c": os.path.join(DATA_DIR, "Outcomes-c.txt"),
}


In [ ]:
def load_data() -> pd.DataFrame:
    chunksize = 10000
    cvs_iter = pd.read_csv(DATA_DIR, chunksize = chunksize)
    chunks = []

    with pd.read_csv("tmp.csv", chunksize=4) as reader:
      for chunk in reader:
        # TODO preprocess data
        # process(chunk: pd.DataFrame) -> pd.DataFrame

    return pd.concat(chunk_list)



    # Normalize possible column name variants
    # Common columns: RecordID, SAPS-I, SOFA, Length_of_stay, Survival, In-hospital_death
    # We'll keep RecordID + In-hospital_death
    # col_candidates = [c for c in df.columns if c.lower().replace("_","-") in ["in-hospital-death", "in-hospital_death", "inhospitaldeath"]]
    # if len(col_candidates) == 0:
    #     raise ValueError(f"Could not find mortality column in {outcome_path}. Columns: {df.columns.tolist()}")
    # death_col = col_candidates[0]
    # return df[["RecordID", death_col]].rename(columns={death_col: "label"})


In [ ]:
def time_to_hour(t: str) -> int:
    # "HH:MM" -> int hour bin (floor)
    hh, mm = t.split(":")
    return int(hh)  # floor to hour


In [ ]:
VITALS = ["HR", "SysABP", "DiasABP", "MAP", "RespRate", "Temp", "SpO2"] # Also user prompts

STATIC = ["Age", "Gender", "Height", "Weight", "ICUType"]  # appear at time 00:00 usually

KEEP_PARAMS = set(VITALS + STATIC)


In [ ]:
CLIP_RANGES = {
    "HR": (20, 250),
    "SysABP": (40, 250),
    "DiasABP": (20, 150),
    "MAP": (30, 200),
    "RespRate": (4, 80),
    "Temp": (25, 45),     # Celsius (often)
    "SpO2": (0, 100),
    "Age": (0, 120),
    "Height": (50, 250),  # cm
    "Weight": (1, 400),   # kg
    "Gender": (0, 1),
    "ICUType": (1, 4),
}

In [ ]:
def load_patient_file(path: str, max_hours: int = 48) -> tuple[int, pd.DataFrame, dict]:
    """
    Returns:
      record_id, hourly_df (index=hour 0..max_hours), static_dict
    """
    record_id = int(os.path.splitext(os.path.basename(path))[0])

    df = pd.read_csv(path)
    # df columns typically: Time, Parameter, Value
    df["Hour"] = df["Time"].astype(str).apply(time_to_hour)
    df["Parameter"] = df["Parameter"].astype(str)

    # Keep only selected params
    df = df[df["Parameter"].isin(KEEP_PARAMS)].copy()

    # Convert to numeric
    df["Value"] = pd.to_numeric(df["Value"], errors="coerce")

    # Clip outliers (optional but recommended)
    for p, (lo, hi) in CLIP_RANGES.items():
        mask = df["Parameter"] == p
        if mask.any():
            df.loc[mask, "Value"] = df.loc[mask, "Value"].clip(lo, hi)

    # Static: pick first non-null value for each static param
    static = {}
    for p in STATIC:
        sub = df[df["Parameter"] == p]["Value"].dropna()
        static[p] = float(sub.iloc[0]) if len(sub) else np.nan

    # Time-varying: aggregate within each hour
    # If multiple values within hour: take LAST (or MEAN). We'll use LAST for vitals.
    tv = df[df["Parameter"].isin(VITALS)].copy()
    tv.sort_values(["Hour"], inplace=True)  # order within hour
    tv_agg = tv.groupby(["Hour", "Parameter"])["Value"].last().reset_index()

    # Pivot to wide: columns = VITALS, index = Hour
    hourly = tv_agg.pivot(index="Hour", columns="Parameter", values="Value")

    # Reindex to full hour grid 0..max_hours (inclusive)
    hourly = hourly.reindex(range(0, max_hours + 1))

    return record_id, hourly, static

In [ ]:
sample_file = glob.glob(os.path.join(DATA_DIR, "set-a", "*.txt"))[0]
rid, hourly_df, static = load_patient_file(sample_file)
rid, hourly_df.head(), static

IndexError: list index out of range

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
def build_dataset(data_dir: str, sets=("set-a","set-b","set-c"), max_hours=48):
    all_records = []
    for s in sets:
        files = sorted(glob.glob(os.path.join(data_dir, s, "*.txt")))
        for f in files:
            rid, hourly, static = load_patient_file(f, max_hours=max_hours)
            all_records.append((rid, hourly, static))

    # Merge with outcomes
    y_map = outcomes_df.set_index("RecordID")["label"].to_dict()

    rows_seq = []
    rows_static = []
    labels = []
    record_ids = []

    for rid, hourly, static in all_records:
        if rid not in y_map:
            continue
        record_ids.append(rid)
        labels.append(int(y_map[rid]))
        rows_seq.append(hourly[VITALS].to_numpy(dtype=np.float32))
        rows_static.append([static[p] for p in STATIC])

    X_seq = np.stack(rows_seq, axis=0)          # (N, T, F)
    X_static = np.array(rows_static, dtype=np.float32)
    y = np.array(labels, dtype=np.int64)
    record_ids = np.array(record_ids, dtype=np.int64)

    return record_ids, X_seq, X_static, y

In [ ]:
record_ids, X_seq, X_static, y = build_dataset(DATA_DIR, sets=SETS, max_hours=48)
X_seq.shape, X_static.shape, y.shape, y.mean()

In [ ]:
def forward_fill_3d(X: np.ndarray) -> np.ndarray:
    """
    X: (N, T, F) with NaNs
    Forward-fill along time axis for each patient & feature.
    """
    X_ff = X.copy()
    N, T, F = X_ff.shape
    for i in range(N):
        for f in range(F):
            last = np.nan
            for t in range(T):
                if np.isnan(X_ff[i, t, f]):
                    X_ff[i, t, f] = last
                else:
                    last = X_ff[i, t, f]
    return X_ff

In [ ]:
def median_impute_3d(X: np.ndarray, medians: np.ndarray = None):
    """
    If medians not provided, compute over all non-NaN values in X.
    """
    X_out = X.copy()
    if medians is None:
        medians = np.nanmedian(X_out.reshape(-1, X_out.shape[-1]), axis=0)
    # Replace NaNs
    inds = np.where(np.isnan(X_out))
    X_out[inds] = np.take(medians, inds[2])
    return X_out, medians

In [ ]:
def missing_indicators(X: np.ndarray) -> np.ndarray:
    # 1 where missing, 0 otherwise
    return np.isnan(X).astype(np.float32)

In [ ]:
X_miss = missing_indicators(X_seq)
X_ff = forward_fill_3d(X_seq)
X_imp, seq_medians = median_impute_3d(X_ff)

# Optionally concatenate indicators as extra channels
X_final = np.concatenate([X_imp, X_miss], axis=2)  # (N, T, 2F)
X_final.shape

In [ ]:
# Fill static NaNs with column median
static_medians = np.nanmedian(X_static, axis=0)
X_static_imp = X_static.copy()
for j in range(X_static.shape[1]):
    mask = np.isnan(X_static_imp[:, j])
    X_static_imp[mask, j] = static_medians[j]

In [ ]:
from sklearn.model_selection import train_test_split

idx = np.arange(len(y))

train_idx, test_idx = train_test_split(idx, test_size=0.10, random_state=42, stratify=y)
train_idx, val_idx  = train_test_split(train_idx, test_size=0.1111, random_state=42, stratify=y[train_idx])
# (0.1111 of 0.90 ≈ 0.10) => ~80/10/10 overall

X_train, X_val, X_test = X_final[train_idx], X_final[val_idx], X_final[test_idx]
S_train, S_val, S_test = X_static_imp[train_idx], X_static_imp[val_idx], X_static_imp[test_idx]
y_train, y_val, y_test = y[train_idx], y[val_idx], y[test_idx]

In [ ]:
def fit_standardizer_3d(X: np.ndarray):
    # X: (N, T, F)
    flat = X.reshape(-1, X.shape[-1])
    mean = flat.mean(axis=0)
    std = flat.std(axis=0) + 1e-8
    return mean, std

def apply_standardizer_3d(X: np.ndarray, mean: np.ndarray, std: np.ndarray):
    return (X - mean) / std

In [ ]:
seq_mean, seq_std = fit_standardizer_3d(X_train)
X_train_sc = apply_standardizer_3d(X_train, seq_mean, seq_std)
X_val_sc   = apply_standardizer_3d(X_val, seq_mean, seq_std)
X_test_sc  = apply_standardizer_3d(X_test, seq_mean, seq_std)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler_static = StandardScaler()
S_train_sc = scaler_static.fit_transform(S_train)
S_val_sc   = scaler_static.transform(S_val)
S_test_sc  = scaler_static.transform(S_test)

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
